# С замером времени инференса

In [ ]:
import gradio as gr
import json
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
from datetime import datetime
import argparse
import logging
import threading
import torch
import time  # ← добавлено для замера времени
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel, PeftConfig

# Настройка логирования
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('gradio_app.log'),
        logging.StreamHandler()
    ]
)

CUSTOM_CSS = """
body {
    zoom: 1.25;
}
"""

# ------------------ Глобальные настройки ------------------
LOG_DIR = "gradio_logs"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_LOCK = threading.Lock()

def append_jsonl(path, entry):
    line = json.dumps(entry, ensure_ascii=False)
    with LOG_LOCK:
        with open(path, "a", encoding="utf-8") as f:
            f.write(line + "\n")
            f.flush()

# ------------------ Конфигурация модели LLM (QLoRA) ------------------
LLM_MODEL_DIR = "./classifiers/apple/apple_SimpleSD-4B-instruct"
BASE_MODEL = None
DIGIT2CLASS = {
    "1": "теоретический вопрос",
    "2": "анализ данных",
    "3": "взаимодействие с графом",
    "4": "анализ данных и взаимодействие с графом",
    "5": "чушь"
}
CLASS_NAMES = list(DIGIT2CLASS.values())
MAX_LENGTH = 620

# ------------------ Конфигурация модели BERT (с LoRA) ------------------
BERT_ADAPTER_DIR = "./classifiers/ai-forever/ai-forever_ru-en-RoSBERTa"
BERT_BASE_MODEL_NAME = "ai-forever/ru-en-RoSBERTa"
BERT_MAX_LENGTH = 620

def load_bert_model():
    global BERT_TOKENIZER, BERT_MODEL, BERT_DEVICE
    logging.info("Загрузка BERT модели с LoRA...")

    BERT_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    BERT_TOKENIZER = AutoTokenizer.from_pretrained(BERT_ADAPTER_DIR)
    if BERT_TOKENIZER.pad_token is None:
        BERT_TOKENIZER.pad_token = BERT_TOKENIZER.eos_token

    base_model = AutoModelForSequenceClassification.from_pretrained(
        BERT_BASE_MODEL_NAME,
        num_labels=5,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

    base_model = base_model.to(BERT_DEVICE)
    BERT_MODEL = PeftModel.from_pretrained(base_model, BERT_ADAPTER_DIR)
    BERT_MODEL.config.pad_token_id = BERT_TOKENIZER.pad_token_id
    BERT_MODEL = BERT_MODEL.to(BERT_DEVICE)
    BERT_MODEL.to(BERT_DEVICE)
    BERT_MODEL.eval()

    logging.info("BERT модель с LoRA загружена на CPU")

def bert_predict(text: str) -> str:
    if BERT_MODEL is None:
        return "BERT не загружен"

    inputs = BERT_TOKENIZER(
        text,
        truncation=True,
        padding=True,
        max_length=510,
        return_tensors="pt"
    )
    inputs = {k: v.to(BERT_DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = BERT_MODEL(**inputs)
        logits = outputs.logits[0]
        probs = torch.softmax(logits, dim=-1)
        pred_id = torch.argmax(probs).item()

    predicted_class = CLASS_NAMES[pred_id] if pred_id < len(CLASS_NAMES) else "неизвестный"
    max_prob = probs[pred_id].item()
    return f"🏷️ Класс: **{predicted_class}**\n\nВероятность: **{max_prob:.4f}**"

# ------------------ Загрузка LLM (QLoRA) ------------------
def load_llm_model(model_dir, base_model_name=None):
    logging.info(f"Загрузка LLM из {model_dir}...")

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if base_model_name is None:
        peft_config = PeftConfig.from_pretrained(model_dir)
        base_model_name = peft_config.base_model_name_or_path
        logging.info(f"Базовая модель LLM: {base_model_name}")

    llm_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float16 if llm_device.type == "cuda" else torch.float32

    base_model = AutoModelForSequenceClassification.from_pretrained(
        base_model_name,
        num_labels=5,
        torch_dtype=dtype,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )

    model = PeftModel.from_pretrained(base_model, model_dir)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model.to(llm_device)
    model.eval()

    logging.info(f"LLM модель загружена на {llm_device}")
    return tokenizer, model

def llm_predict(query, tokenizer, model, return_probs=False):
    inputs = tokenizer(
        query,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    use_amp = device.type == "cuda"

    with torch.no_grad():
        if use_amp:
            with torch.cuda.amp.autocast(enabled=True):
                outputs = model(**inputs)
        else:
            outputs = model(**inputs)

        logits = outputs.logits[0]
        probs = torch.softmax(logits, dim=-1)
        pred_id = torch.argmax(probs).item()

    predicted_class = CLASS_NAMES[pred_id]
    if return_probs:
        return predicted_class, dict(zip(CLASS_NAMES, probs.detach().cpu().numpy()))
    return predicted_class

# ------------------ Глобальная загрузка моделей ------------------
llm_tokenizer, llm_model = None, None
BERT_TOKENIZER, BERT_MODEL, BERT_DEVICE = None, None, torch.device("cpu")

try:
    llm_tokenizer, llm_model = load_llm_model(LLM_MODEL_DIR, BASE_MODEL)
except Exception:
    logging.exception("Ошибка при загрузке LLM")

try:
    load_bert_model()
except Exception:
    logging.exception("Ошибка при загрузке BERT")

# ------------------ Логирование ------------------
def log_interaction(model_name, input_text, output_text, elapsed=None):  # ← параметр elapsed
    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "model": model_name,
        "input": input_text,
        "output": output_text,
    }
    if elapsed is not None:                    # ← добавляем время, если передано
        log_entry["elapsed_time_sec"] = round(elapsed, 4)

    append_jsonl(os.path.join(LOG_DIR, "interactions.jsonl"), log_entry)
    logging.info(f"Запрос к {model_name} сохранён")

# ------------------ Основная функция обработки ------------------
def process_query(text, request: gr.Request):
    """Обрабатывает запрос: запускает LLM и BERT, возвращает ТОЛЬКО ответ LLM."""
    if not text or not text.strip():
        return "Пустой запрос", "", ""

    if llm_tokenizer is None or llm_model is None:
        return "LLM не загружена", text, ""

    # --- Замер LLM ---
    t0 = time.perf_counter()
    try:
        llm_class, llm_probs = llm_predict(text, llm_tokenizer, llm_model, return_probs=True)
        top_prob = llm_probs[llm_class]
        llm_visible = f"🏷️ Класс: **{llm_class}**\n\nВероятность: **{top_prob * 100:.0f}%**"
    except Exception as e:
        llm_visible = f"Ошибка LLM: {str(e)}"
        logging.error(f"LLM inference error: {e}")
    t_llm = time.perf_counter() - t0

    # --- Замер BERT ---
    t0 = time.perf_counter()
    try:
        bert_result = bert_predict(text)
    except Exception as e:
        bert_result = f"BERT error: {str(e)}"
        logging.error(f"BERT inference error: {e}")
    t_bert = time.perf_counter() - t0

    # Логируем с временем выполнения
    log_interaction("llm", text, llm_visible, elapsed=t_llm)
    log_interaction("bert", text, bert_result, elapsed=t_bert)

    # (опционально) можно добавить время в вывод, но по условию не трогаем
    return llm_visible, text, llm_visible

# ------------------ Построение интерфейса ------------------
def build_demo():
    with gr.Blocks(
        title="Помогите нам улучшить модель — вводите запросы и сообщайте об ошибках!",
        css=CUSTOM_CSS
    ) as demo:
        gr.Markdown("""## 🧪 Тестирование классификатора запросов

        Для тестирования вам предложен классификатор запросов (определяет, к какому классу отновится запрос) для графовой no-code платформы [Smile.Cloud](https://iai.itmo.ru/razrabotki/texnologii-i-sredstva-razrabotki-sistem-ii/smile.cloud-platforma-dlya-byistrogo-prototipirovaniya,-razrabotki-i-obucheniya-modelej-na-dannyix).

        **Существует всего 5 классов запросов, которые обрабатываются классификатором:**
        1. Теоретический вопрос - вопросы о принципах ML, математике, статистике, алгоритмах.
        2. Анализ данных - работа с таблицами: EDA, визуализация, фильтрация, группировки, очистка, расчёт статистик.
        3. Взаимодействие с графом - построение пайплайна на графе, работа с ML моделями и содержанием графа.
        4. Анализ данных и взаимодействие с графом - запрос, который требует и анализа табличных данных, и операций с графом.
        5. Чушь - всё, что не относится к ML, анализу данных, графам, математике.

        **Задача тестировщика:**
        1) Вводить самые разные запросы - короткие, длинные, с опечатками, с разговорными формулировками...
        2) Попытайтесь запутать модель: используйте смешанные классы, двусмысленные фразы, сленг.
        3) Для каждого запроса мысленно определите один класс, который, по вашему мнению, правильный.
        4) Если модель выдала другой класс - отметьте ожидаемый вами класс и нажмите «Сообщить об ошибке» и/или напишите комментарий, какой класс вы ожидали и почему. При неоднозначности тоже отмечайте.
        5) Ошибкой считается несовпадение с вашим ожидаемым классом или явно некорректный ответ модели (например, на «как дела?» выдала «анализ данных»).
        """)

        with gr.Accordion("📋 Примеры запросов для каждого класса (нажмите, чтобы развернуть)", open=False):
            gr.Markdown("""
        - **теоретический вопрос**: «Что такое градиентный бустинг?», «Объясни разницу между L1 и L2 регуляризацией», «регуляризация», «логистическая регресия сигмоида формула зачем нужна она там вообще», «объясните SVM».
        - **анализ данных**:  «построй гистограмму цен», «удали дубликаты», «Удали пропуски в столбце age», «построй heatmap матрицы корреляций», «Разбей юзеров на группы по возрасту, вычисли их суммарный доход и выведи только тех, кто принес больше миллиона».
        - **взаимодействие с графом**:  «добавь xgboost», «Настрой early stopping с patience=10, отслеживая валидационный loss», «Запусти обучение модели», «Установи min_child_weight=5 и gamma=0.2 в блоке градиентного бустинга», «create pipeline: data, scale, clf (logreg)».
        - **анализ данных и взаимодействие с графом**: «Обработай категориальные признаки и построй график», «Открой CSV, построй график распределения и добавь слой нормализации в пайплайн», «Предобработай данные и добавь RandomForest в граф», «Отфильтруй транзакции выше 1000, посчитай их долю по дням и добавь это как узел feature engineering в граф перед моделью градиентного бустинга»
        - **чушь**: «Как сварить пельмени?», «Как оптимизировать жизнь через hyperparameters?», «Соедини узлы по случайному критерию», «обучи модель без данных», «ывапку345».

        """)

        with gr.Row():
            inp = gr.Textbox(
                label="Ваш запрос",
                lines=2,
                scale=4,
                placeholder="Введите ваш запрос здесь..."
            )
            btn = gr.Button("🚀 Классифицировать", variant="primary", scale=1)
            out = gr.Markdown(
                label="Результат",
                value="*Результат появится здесь...*"
            )

        last_query = gr.State("")
        last_prediction = gr.State("")

        gr.Markdown("### 🚩 Сообщить об ошибке")
        with gr.Row():
            correct_class = gr.Radio(
                choices=[
                    "теоретический вопрос",
                    "анализ данных",
                    "взаимодействие с графом",
                    "анализ данных и взаимодействие с графом",
                    "чушь"
                ],
                label="Какой класс вы ожидали?",
                interactive=True,
                scale=2
            )
            feedback_comment = gr.Textbox(
                label="Комментарий (необязательно)",
                lines=1,
                scale=1
            )
            feedback_btn = gr.Button("📩 Отправить замечание", size="sm", scale=0, min_width=80)

        btn.click(
            process_query,
            inputs=inp,
            outputs=[out, last_query, last_prediction]
        )
        feedback_btn.click(
            handle_feedback,
            inputs=[last_query, last_prediction, correct_class, feedback_comment],
            outputs=[]
        )

    demo.queue(default_concurrency_limit=4, max_size=256)
    return demo

def handle_feedback(query, prediction, correct_class, comment):
    if not correct_class:
        gr.Warning("⚠️ Выберите класс, который вы ожидали")
        return

    entry = {
        "timestamp": datetime.now().isoformat(),
        "query": query,
        "prediction": prediction,
        "expected_class": correct_class,
        "comment": comment,
    }

    append_jsonl(os.path.join(LOG_DIR, "flags.jsonl"), entry)
    logging.info(f"Жалоба сохранена: ожидал {correct_class}")
    gr.Info("😊 Спасибо! Ваше замечание поможет улучшить модель.")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--port", type=int, default=8891)
    parser.add_argument("--share", action="store_true")
    args = parser.parse_args()

    logging.info(f"Запуск Gradio на порту {args.port}, share={args.share}")
    demo = build_demo()
    demo.launch(
        server_name="0.0.0.0",
        server_port=args.port,
        share=False,
        app_kwargs={"description": "Помогите нам улучшить модель — вводите запросы и сообщайте об ошибках!"},
        show_error=True,
        root_path=""
    )